In [ ]:
# PLM Abortion Example - Python Conversion
#
# Parallel to R Code/ExampleFEPLM.R and Stata Code/DLfePLM.do. Reproduces the
# Day-2 "Abortion and Crime" (PLM with fixed effects) results: baseline TWFE,
# flexible-trend "kitchen sink" TWFE, post-double-selection (BCHK clustered
# loadings), and CV partialing-out. Standard errors are clustered on state.

######################### Libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import norm
from patsy import dmatrix
import os
import warnings
warnings.filterwarnings('ignore')

######################### Options / data
# Resolve the data path whether launched from the repo root or "Python code/".
_data_path = os.path.join("Data", "levitt_ex.dat")
if not os.path.exists(_data_path):
    _data_path = os.path.join("..", "Data", "levitt_ex.dat")
data_DL = pd.read_csv(_data_path, sep="\t")   # tab-delimited (matches R loader)

# Clean out some observations
data_DL = data_DL[data_DL.iloc[:, 0] != 9]  # drop DC observations
data_DL = data_DL[data_DL.iloc[:, 0] != 2]  # drop Alaska observations
data_DL = data_DL[data_DL.iloc[:, 0] != 12]  # drop Hawaii observations

# Restrict to years in original Donohue and Levitt paper
data_DL = data_DL[(data_DL.iloc[:, 1] >= 85) & (data_DL.iloc[:, 1] <= 97)]

# We're going to use a trend later, and let's have it start at 1
data_DL.iloc[:, 1] = data_DL.iloc[:, 1] - 84

# Treat state and year as factors (assuming column names are 'statenum' and 'year')
data_DL['statenum'] = data_DL['statenum'].astype('category')
data_DL['year'] = data_DL['year'].astype('category')

# State and year dummies
DS = pd.get_dummies(data_DL['statenum'], drop_first=False).values
DY = pd.get_dummies(data_DL['year'], drop_first=False).values

# Create model matrix for state and year dummies
DYS = dmatrix('statenum + year', data=data_DL, return_type='dataframe')
DYS = DYS.values

# Scale some of the variables
data_DL['xxincome'] = data_DL['xxincome'] / 100
data_DL['xxpover'] = data_DL['xxpover'] / 100
data_DL['xxafdc15'] = data_DL['xxafdc15'] / 10000
data_DL['xxbeer'] = data_DL['xxbeer'] / 100

# Cluster-robust SE on state (mirrors R's vcovCL(..., cluster = statenum))
clust = data_DL['statenum']

############################################################
# Baseline TWFE -- see if we can get close to original DL results

# Violent crime
formula_viol = 'lpc_viol ~ efaviol + xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)'
lm_viol = ols(formula_viol, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Viol: {lm_viol.params['efaviol']}")
print(f"s.e. FE Viol: {lm_viol.bse['efaviol']}")

# Property crime
formula_prop = 'lpc_prop ~ efaprop + xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)'
lm_prop = ols(formula_prop, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Prop: {lm_prop.params['efaprop']}")
print(f"s.e. FE Prop: {lm_prop.bse['efaprop']}")

# Murder
formula_murd = 'lpc_murd ~ efamurd + xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)'
lm_murd = ols(formula_murd, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Murd: {lm_murd.params['efamurd']}")
print(f"s.e. FE Murd: {lm_murd.bse['efamurd']}")


##################################################################
# Initial conditions and within state means
var_names = ['efaviol', 'efaprop', 'efamurd', 'xxprison', 'xxpolice',
             'xxunemp', 'xxincome', 'xxpover', 'xxafdc15', 'xxgunlaw', 'xxbeer']

# Initial conditions
x0 = data_DL[data_DL['year'] == 1][var_names].values
X0 = DS @ x0
X0_df = pd.DataFrame(X0, columns=['efaviol0', 'efaprop0', 'efamurd0', 'xxprison0', 'xxpolice0',
                                  'xxunemp0', 'xxincome0', 'xxpover0', 'xxafdc150', 'xxgunlaw0', 'xxbeer0'])

# State means
XB = DS @ np.linalg.solve(DS.T @ DS, DS.T @ data_DL[var_names].values)
XB_df = pd.DataFrame(XB, columns=['efaviolB', 'efapropB', 'efamurdB', 'xxprisonB', 'xxpoliceB',
                                  'xxunempB', 'xxincomeB', 'xxpoverB', 'xxafdc15B', 'xxgunlawB', 'xxbeerB'])

# Combine to raw data
data_DL = pd.concat([data_DL.reset_index(drop=True), X0_df, XB_df], axis=1)
clust = data_DL['statenum']   # refresh aligned cluster id after reset_index

####################################################################
# Make dictionary expansion

# Go back to treating year as numeric for a smooth deterministic trend
data_DL['year'] = pd.to_numeric(data_DL['year'])
data_DL['year'] = data_DL['year'] / data_DL['year'].max()  # scale to unit interval
data_DL['year2'] = data_DL['year'] ** 2
data_DL['year3'] = data_DL['year'] ** 3

x_names = ['xxprison', 'xxpolice', 'xxunemp', 'xxincome', 'xxpover', 'xxafdc15', 'xxgunlaw', 'xxbeer']
x0_names = ['xxprison0', 'xxpolice0', 'xxunemp0', 'xxincome0', 'xxpover0', 'xxafdc150', 'xxgunlaw0', 'xxbeer0']
xB_names = ['xxprisonB', 'xxpoliceB', 'xxunempB', 'xxincomeB', 'xxpoverB', 'xxafdc15B', 'xxgunlawB', 'xxbeerB']

# Build control formulas
controls_viol = (f"{' + '.join(x_names)} + " +
                f"({' + '.join(x_names)}):(year + year2 + year3) + " +
                f"(efaviol0 + {' + '.join(x0_names)} + efaviolB + {' + '.join(xB_names)}):(year + year2 + year3)")

controls_prop = (f"{' + '.join(x_names)} + " +
                f"({' + '.join(x_names)}):(year + year2 + year3) + " +
                f"(efaprop0 + {' + '.join(x0_names)} + efapropB + {' + '.join(xB_names)}):(year + year2 + year3)")

controls_murd = (f"{' + '.join(x_names)} + " +
                f"({' + '.join(x_names)}):(year + year2 + year3) + " +
                f"(efamurd0 + {' + '.join(x0_names)} + efamurdB + {' + '.join(xB_names)}):(year + year2 + year3)")

# Create design matrices
X_viol = dmatrix(f"~ {controls_viol}", data=data_DL, return_type='dataframe')
X_viol = X_viol.iloc[:, 1:].values  # Remove intercept

X_prop = dmatrix(f"~ {controls_prop}", data=data_DL, return_type='dataframe')
X_prop = X_prop.iloc[:, 1:].values  # Remove intercept

X_murd = dmatrix(f"~ {controls_murd}", data=data_DL, return_type='dataframe')
X_murd = X_murd.iloc[:, 1:].values  # Remove intercept

# Partial out state and year dummies
proj_matrix = DYS @ np.linalg.solve(DYS.T @ DYS, DYS.T)

y_viol = data_DL['lpc_viol'].values - proj_matrix @ data_DL['lpc_viol'].values
d_viol = data_DL['efaviol'].values - proj_matrix @ data_DL['efaviol'].values
X_viol = X_viol - proj_matrix @ X_viol

y_prop = data_DL['lpc_prop'].values - proj_matrix @ data_DL['lpc_prop'].values
d_prop = data_DL['efaprop'].values - proj_matrix @ data_DL['efaprop'].values
X_prop = X_prop - proj_matrix @ X_prop

y_murd = data_DL['lpc_murd'].values - proj_matrix @ data_DL['lpc_murd'].values
d_murd = data_DL['efamurd'].values - proj_matrix @ data_DL['efamurd'].values
X_murd = X_murd - proj_matrix @ X_murd

############################################################################
# TWFE with flexible trends ("kitchen sink" / All Controls)

# Violent crime
formula_ft_viol = f"lpc_viol ~ efaviol + {controls_viol} + C(statenum) + C(year)"
lm_ft_viol = ols(formula_ft_viol, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Viol (Flexible): {lm_ft_viol.params['efaviol']}")
print(f"s.e. FE Viol (Flexible): {lm_ft_viol.bse['efaviol']}")

# Property crime
formula_ft_prop = f"lpc_prop ~ efaprop + {controls_prop} + C(statenum) + C(year)"
lm_ft_prop = ols(formula_ft_prop, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Prop (Flexible): {lm_ft_prop.params['efaprop']}")
print(f"s.e. FE Prop (Flexible): {lm_ft_prop.bse['efaprop']}")

# Murder
formula_ft_murd = f"lpc_murd ~ efamurd + {controls_murd} + C(statenum) + C(year)"
lm_ft_murd = ols(formula_ft_murd, data=data_DL).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Murd (Flexible): {lm_ft_murd.params['efamurd']}")
print(f"s.e. FE Murd (Flexible): {lm_ft_murd.bse['efamurd']}")

# Sample size and number of regressors
print(f"Sample size: {len(data_DL)}")
print(f"Number of noncollinear regressors (Violent Crime): {lm_ft_viol.df_model}")


############################################################################
# Double selection with clustered loadings (BCHK plug-in penalty)

def cl_lasso(X, y, cluster_dummies, init_resid):
    """Lasso with the BCHK plug-in penalty and cluster-robust loadings.

    Mirrors the R cl.lasso(): lambda = 2.2*sqrt(n)*Phi^{-1}(1 - (.1/log n)/(2p)),
    scores scaled by the within-cluster sd of the score. The BCHK penalty is
    calibrated to the *summed* loss; sklearn's Lasso minimizes the *mean* loss
    (1/2n)||y-Xb||^2 + alpha||b||_1, so the equivalent regularization is lam/n
    (this matches the variable selection R's glmnet returns at lambda=lam).
    """
    n, p = X.shape

    # Score and within-cluster sum
    Syx = X * init_resid.reshape(-1, 1)
    DSyx = cluster_dummies.T @ Syx
    Ups0 = np.sqrt(np.sum(DSyx ** 2, axis=0)) / n   # sd of score

    # BCHK plug-in penalty (deterministic; uses the normal quantile)
    lam = 2.2 * np.sqrt(n) * norm.ppf(1 - (0.1 / np.log(n)) / (2 * p))

    scaledX = X / Ups0
    lasso = Lasso(alpha=lam / n, fit_intercept=False)   # mean-loss scaling
    lasso.fit(scaledX, y)
    return lasso.coef_

# Helper function to get initial residuals
def get_init_resid(formula, data):
    model = ols(formula, data=data).fit()
    return model.resid.values

# Double selection - Violent crime
init_resid_y_viol = get_init_resid(
    'lpc_viol ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)
init_resid_d_viol = get_init_resid(
    'efaviol ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)

las_y_viol = cl_lasso(X=X_viol, y=y_viol, cluster_dummies=DS, init_resid=init_resid_y_viol)
las_d_viol = cl_lasso(X=X_viol, y=d_viol, cluster_dummies=DS, init_resid=init_resid_d_viol)

use_viol = np.union1d(np.where(las_y_viol != 0)[0], np.where(las_d_viol != 0)[0])

RHS_PDS_viol = np.column_stack([d_viol, X_viol[:, use_viol]])
PDS_viol = sm.OLS(y_viol, RHS_PDS_viol).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE Viol (PDS): {PDS_viol.params[0]}")
print(f"s.e. FE Viol (PDS): {PDS_viol.bse[0]}")

# Double selection - Property crime
init_resid_y_prop = get_init_resid(
    'lpc_prop ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)
init_resid_d_prop = get_init_resid(
    'efaprop ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)

las_y_prop = cl_lasso(X=X_prop, y=y_prop, cluster_dummies=DS, init_resid=init_resid_y_prop)
las_d_prop = cl_lasso(X=X_prop, y=d_prop, cluster_dummies=DS, init_resid=init_resid_d_prop)

use_prop = np.union1d(np.where(las_y_prop != 0)[0], np.where(las_d_prop != 0)[0])

RHS_PDS_prop = np.column_stack([d_prop, X_prop[:, use_prop]])
PDS_prop = sm.OLS(y_prop, RHS_PDS_prop).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE prop (PDS): {PDS_prop.params[0]}")
print(f"s.e. FE prop (PDS): {PDS_prop.bse[0]}")

# Double selection - Murder
init_resid_y_murd = get_init_resid(
    'lpc_murd ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)
init_resid_d_murd = get_init_resid(
    'efamurd ~ xxprison + xxpolice + xxunemp + xxincome + xxpover + xxafdc15 + xxgunlaw + xxbeer + C(statenum) + C(year)',
    data_DL)

las_y_murd = cl_lasso(X=X_murd, y=y_murd, cluster_dummies=DS, init_resid=init_resid_y_murd)
las_d_murd = cl_lasso(X=X_murd, y=d_murd, cluster_dummies=DS, init_resid=init_resid_d_murd)

use_murd = np.union1d(np.where(las_y_murd != 0)[0], np.where(las_d_murd != 0)[0])

RHS_PDS_murd = np.column_stack([d_murd, X_murd[:, use_murd]])
PDS_murd = sm.OLS(y_murd, RHS_PDS_murd).fit(cov_type='cluster', cov_kwds={'groups': clust})
print(f"FE murd (PDS): {PDS_murd.params[0]}")
print(f"s.e. FE murd (PDS): {PDS_murd.bse[0]}")


FE Viol: -0.13044758030605377
s.e. FE Viol: 0.04497637810478967
FE Prop: -0.0910024665215751
s.e. FE Prop: 0.015547937366558042
FE Murd: -0.13054427462797324
s.e. FE Murd: 0.05723029842606615


FE Viol (Flexible): 0.3229282802452045
s.e. FE Viol (Flexible): 0.34188164438073626
FE Prop (Flexible): -0.036318825355772244
s.e. FE Prop (Flexible): 0.06353984188902527
FE Murd (Flexible): 0.8766097941894331
s.e. FE Murd (Flexible): 0.6237115958741819
Sample size: 624
Number of noncollinear regressors (Violent Crime): 146.0


FE Viol (PDS): -0.08259119220838122
s.e. FE Viol (PDS): 0.12947222899145158
FE prop (PDS): -0.04920965781402257
s.e. FE prop (PDS): 0.03679802366196638
FE murd (PDS): 0.012992849983846769
s.e. FE murd (PDS): 0.25877574898217637


FE Viol (CV): 0.02096326953746163
s.e. FE Viol (CV): 0.13154773975193348


FE prop (CV): -0.07064836837719074
s.e. FE prop (CV): 0.04704517770588797


FE murd (CV): -0.06838157608305047
s.e. FE murd (CV): 0.23249162787766425
